<!-- # This script annotates transcripts of video simulation role-playing exercises from job candidates using large language models -->

In [ ]:
# Install libraries that might not be installed as necessary (adjust manually) through Anaconda prompt

In [1]:
# Import necessary libraries
# import openai          # OpenAI API for AI model access
# from openai import OpenAI  # The main client class for interacting with OpenAI's API
from openai import AzureOpenAI
import requests        # HTTP library for making web requests (API calls, downloading data, etc.)
import re              # Regular expressions
import os              # Operating system interface
import pandas as pd    # DataFrames for tabular data manipulation and analysis
import numpy as np     # Numerical computing
import time            # Time-related functions
import random          # Random number generation
import getpass         # pop-up window for API completion
from tqdm import tqdm  # for showing progress bars
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score # for performance metrics
from scipy.stats import pearsonr # for pearson r

# set seed for reproducibility
random.seed(1337)  # Sets seed for Python's random module

In [2]:

# Load data
# test_data = pd.read_excel("input/data_gitp_short_copy_for_prompt_testing.xlsx") # full advisement data
# test_data = pd.read_excel("input/data_gitp - Copy_for_data_exploration.xlsx") # n = 30 advisement participants
# test_data = test_data.iloc[:2] # keep only first few rows for pilot data analysis purposes
# test_data.head()

# Load full dataset
data_gitp = pd.read_csv("input/data_gitp_short.csv")

# Load datasets per job competence: Four transcript versions per competence:
# English = '_en'; english-textwash = '_en_tw'
# Dutch = '_nl', dutch-textwash = '_nl_tw'
# # Advisement 2
advise_en = data_gitp.loc[data_gitp['VideoSimulatieType'] == 'Advisement_2', 
                                 ['id', 'Beoordeling.all_en', 'AD_PROBL', 'AD_CREAT', 'AD_OORDE', 'AD_ORGANS', 'AD_total']]
advise_en_tw = data_gitp.loc[data_gitp['VideoSimulatieType'] == 'Advisement_2', 
                                 ['id', 'Beoordeling.all_en_tw', 'AD_PROBL', 'AD_CREAT', 'AD_OORDE', 'AD_ORGANS', 'AD_total']]
advise_nl = data_gitp.loc[data_gitp['VideoSimulatieType'] == 'Advisement_2', 
                                 ['id', 'Beoordeling.all_nl', 'AD_PROBL', 'AD_CREAT', 'AD_OORDE', 'AD_ORGANS', 'AD_total']]
advise_nl_tw = data_gitp.loc[data_gitp['VideoSimulatieType'] == 'Advisement_2', 
                                 ['id', 'Beoordeling.all_nl_tw', 'AD_PROBL', 'AD_CREAT', 'AD_OORDE', 'AD_ORGANS', 'AD_total']]

# # Change management 2
change_manage_en = data_gitp.loc[data_gitp['VideoSimulatieType'] == 'Change_Management_2', 
                                 ['id', 'Beoordeling.all_en', 'CM_LEIDIN', 'CM_COACH', 'CM_RESUL', 'CM_VISIEU', 'CM_total']]
change_manage_en_tw = data_gitp.loc[data_gitp['VideoSimulatieType'] == 'Change_Management_2', 
                                 ['id', 'Beoordeling.all_en_tw', 'CM_LEIDIN', 'CM_COACH', 'CM_RESUL', 'CM_VISIEU', 'CM_total']]
change_manage_nl = data_gitp.loc[data_gitp['VideoSimulatieType'] == 'Change_Management_2', 
                                 ['id', 'Beoordeling.all_nl', 'CM_LEIDIN', 'CM_COACH', 'CM_RESUL', 'CM_VISIEU', 'CM_total']]
change_manage_nl_tw = data_gitp.loc[data_gitp['VideoSimulatieType'] == 'Change_Management_2', 
                                 ['id', 'Beoordeling.all_nl_tw', 'CM_LEIDIN', 'CM_COACH', 'CM_RESUL', 'CM_VISIEU', 'CM_total']]

# # Team management 2
team_manage_en = data_gitp.loc[data_gitp['VideoSimulatieType'] == 'Team_Management_2', 
                                 ['id', 'Beoordeling.all_en', 'TM_OVERT', 'TM_LEIDIN', 'TM_COACH', 'TM_RESUL', 'TM_total']]
team_manage_en_tw = data_gitp.loc[data_gitp['VideoSimulatieType'] == 'Team_Management_2', 
                                 ['id', 'Beoordeling.all_en_tw', 'TM_OVERT', 'TM_LEIDIN', 'TM_COACH', 'TM_RESUL', 'TM_total']]
team_manage_nl = data_gitp.loc[data_gitp['VideoSimulatieType'] == 'Team_Management_2', 
                                 ['id', 'Beoordeling.all_nl', 'TM_OVERT', 'TM_LEIDIN', 'TM_COACH', 'TM_RESUL', 'TM_total']]
team_manage_nl_tw = data_gitp.loc[data_gitp['VideoSimulatieType'] == 'Team_Management_2', 
                                 ['id', 'Beoordeling.all_nl_tw', 'TM_OVERT', 'TM_LEIDIN', 'TM_COACH', 'TM_RESUL', 'TM_total']]

# # Team work 2
team_work_en = data_gitp.loc[data_gitp['VideoSimulatieType'] == 'Team_Work_2', 
                                 ['id', 'Beoordeling.all_en', 'TW_SAMEN', 'TW_OVERT', 'TW_MONDEC', 'TW_ORGANE', 'TW_total']]
team_work_en_tw = data_gitp.loc[data_gitp['VideoSimulatieType'] == 'Team_Work_2', 
                                 ['id', 'Beoordeling.all_en_tw', 'TW_SAMEN', 'TW_OVERT', 'TW_MONDEC', 'TW_ORGANE', 'TW_total']]
team_work_nl = data_gitp.loc[data_gitp['VideoSimulatieType'] == 'Team_Work_2', 
                                 ['id', 'Beoordeling.all_nl', 'TW_SAMEN', 'TW_OVERT', 'TW_MONDEC', 'TW_ORGANE', 'TW_total']]
team_work_nl_tw = data_gitp.loc[data_gitp['VideoSimulatieType'] == 'Team_Work_2', 
                                 ['id', 'Beoordeling.all_nl_tw', 'TW_SAMEN', 'TW_OVERT', 'TW_MONDEC', 'TW_ORGANE', 'TW_total']]



In [4]:
# import prompts
from prompts.prompt_AD2_en import promptAD_en # Advisement (English)
from prompts.prompt_CM2_en import promptCM_en # Change management (English)
from prompts.prompt_TM2_en import promptTM_en # Team managemement (English)
from prompts.prompt_TW2_en import promptTW_en # Teamwork (English)

from prompts.prompt_AD2_nl import promptAD_nl # Advisement (Dutch)
from prompts.prompt_CM2_nl import promptCM_nl # Change management (Dutch)
from prompts.prompt_TM2_nl import promptTM_nl # Team managemement (Dutch)
from prompts.prompt_TW2_nl import promptTW_nl # Teamwork (Dutch)

# from prompt_test.AD2_expl_en_c_20260108 import promptAD # test prompts
# print("The number of keys in the dictionary is: ",len(promptAD_nl)) # check that it correctly contains 4 'keys'

In [68]:
# # print
promptTW_nl

{'TW_SAMEN': ('TW_SAMEN',
  '\n**Context van de rollenspeloefening:**  \nJe bent een professionele recruiter met uitgebreide ervaring in het beoordelen van functiecompetenties op basis van rollenspeloefeningen.  \nJe taak is om een sollicitant te beoordelen op basis van diens antwoorden in een asynchrone video-rollenspeloefening  \ndie uit vier scènes bestaat.\n\nIn deze rollenspeloefening heeft de kandidaat interactie met zijn/haar collega Marijne, die vriendelijk, energiek en over \nhet algemeen prettig is om mee samen te werken. Twee weken geleden zijn zij begonnen aan een gezamenlijke opdracht \ndie nauwe samenwerking vereist: samen moeten zij een nieuw idee voor de organisatie ontwikkelen, een schriftelijk rapport \nopstellen en een presentatie voorbereiden. De tijd dringt, maar het is hen nog niet gelukt om het werk goed op elkaar af te stemmen \nof te plannen. In de verschillende scènes probeert Marijne herhaaldelijk het grootste deel van de werklast naar de kandidaat te verschu

In [ ]:
# select the first 30 rows for testing
# team_work_en = team_work_en.head(30)

In [ ]:
team_work_nl_tw.shape
# team_manage_en.head

(221, 7)

<!-- # Azure Foundry - API key -->

In [3]:
# Import Azure OpenAI API key
azure_api_key = getpass.getpass("Enter API key for Azure OpenAI: ") # manually enter API key to the pop-up window

# Import Azure OpenAI endpoint
azure_endpoint = getpass.getpass("Enter Azure OpenAI endpoint URL: ").strip() # manually enter endpoint URL

api_version = "2024-12-01-preview"

client = AzureOpenAI(
    api_version=api_version,
    azure_endpoint=azure_endpoint,
    api_key=azure_api_key,
)

<!-- ## Azure Foundry - Function to get rating from OpenAI -->

In [7]:
# Set default parameters
# model = "gpt-4.1-mini" # manually change the model
deployment = "gpt-4.1-mini" # manually change the model # This is the custom name you gave in Azure AI Studio, e.g., "my-gpt4-deployment"
temperature = 1      # manually change the temperature

# A - Function for explanation (number + explanation in the same cell)
def get_explanation(client, text, prompt):
    """Sends the text to GPT (Azure) and returns both score and explanation."""
    try:
        full_prompt = prompt.format(text=text)

        response = client.chat.completions.create(
            model=deployment,  # Azure deployment name
            messages=[
                {"role": "system", "content": "You are a professional recruiter."},
                {"role": "user", "content": full_prompt}
            ],
            temperature=temperature,
            max_completion_tokens=4000
        )

        return response.choices[0].message.content.strip()

    except Exception as e:
        print("Error:", e)
        return ""

# B - Function for numerical rating (1-5)
def get_rating(client, text, prompt):
    """Sends the text to GPT (Azure) and returns a numeric score (1–5, decimals allowed)."""
    try:
        full_prompt = prompt.format(text=text)

        response = client.chat.completions.create(
            model=deployment,
            messages=[
                {"role": "system", "content": "You are a professional recruiter."},
                {"role": "user", "content": full_prompt}
            ],
            temperature=temperature,
            max_completion_tokens=500
        )

        reply = response.choices[0].message.content.strip()

        import re
        match = re.search(r"\d+(\.\d+)?", reply)
        if match:
            return float(match.group())
        else:
            print("Warning: No numeric value found in reply:", reply)
            return None

    except Exception as e:
        print("Error:", e)
        return None

In [78]:
# Create empty dataframe to store the annotation results + explanation ONLY (but not numerical score)
# sed default parameters
df = team_work_nl_tw # MANUALLY change dataset as necessary
prompt_dict = promptTW_nl # MANUALLY change prompt as necessary
version = "nl_tw" # MANUALLY set language and anonymization status
model_short = "gpt4.1"

for code in prompt_dict.keys():
    df[f"{model_short}_{code}_{version}_explanation"] = None 
    # df[f"{model_short}_{code}_score"] = None

print(df.columns) # make sure that correct empty columns have been created

Index(['id', 'Beoordeling.all_nl_tw', 'TW_SAMEN', 'TW_OVERT', 'TW_MONDEC',
       'TW_ORGANE', 'TW_total', 'gpt4.1_TW_SAMEN_nl_tw_explanation',
       'gpt4.1_TW_OVERT_nl_tw_explanation',
       'gpt4.1_TW_MONDEC_nl_tw_explanation',
       'gpt4.1_TW_ORGANE_nl_tw_explanation'],
      dtype='object')


<!-- # Text annotation -->

In [79]:
# Annotate ONLY verbal explanation + numerical value
# Manually set default parameters
df = team_work_nl_tw # MANUALLY change dataset as necessary
prompt_dict = promptTW_nl # MANUALLY change prompt as necessary 
annotation_column = 'Beoordeling.all_nl_tw' # MANUALLY change as necessary
version = "nl_tw" # MANUALLY set language and anonymization status
model_short = "gpt4.1" # manually change model name as necessary

# Loop through all rows
for i, row in tqdm(df.iterrows(), total=len(df)):
    text = row[annotation_column]
    for code, (trait, prompt_expl) in prompt_dict.items():
        expl = get_explanation(client, text, prompt_expl)
        df.at[i, f"{model_short}_{code}_{version}_explanation"] = expl

100%|██████████| 221/221 [21:44<00:00,  5.90s/it]


In [80]:
# Preview annotated df
print(df.head())

        id                              Beoordeling.all_nl_tw  TW_SAMEN  \
1  1462073  Marine, ja, ik begrijp inderdaad dat jij het d...  3.000000   
3  1552130  Hoi marine, leuk om met je samen te werken. Zo...  3.500000   
5  1584726  Marine, wat goed dat we elkaar inderdaad treff...  3.501667   
6  1584752  Hoi marine, bedankt voor je compliment dat je ...  3.500000   
7  1586157  Ik snap dat je het heel druk hebt. Dat hebben ...  3.666667   

   TW_OVERT  TW_MONDEC  TW_ORGANE  TW_total  \
1  3.000000   3.000000   3.000000  3.000000   
3  3.001667   2.831667   3.333333  3.166667   
5  3.666667   3.001667   3.833333  3.500833   
6  3.166667   3.166667   3.500000  3.333333   
7  2.998333   3.001667   3.335000  3.250417   

                   gpt4.1_TW_SAMEN_nl_tw_explanation  \
1  Score: 4.0 - Toelichting: De kandidaat toont s...   
3  Score: 4.0 - Toelichting: De kandidaat toont s...   
5  Score: 4.0 - Toelichting: De kandidaat toont c...   
6  Score: 3.5 - Toelichting: De kandidaat 

In [81]:
# Extract the numerical value from the annotation and store in new columns
# MANUALLY List of existing explanation columns
explanation_cols = ['gpt4.1_TW_SAMEN_nl_tw_explanation',
       'gpt4.1_TW_OVERT_nl_tw_explanation',
       'gpt4.1_TW_MONDEC_nl_tw_explanation',
       'gpt4.1_TW_ORGANE_nl_tw_explanation']

# Names of the new numeric columns to add
numeric_cols = ['gpt4.1_TW_SAMEN_nl_tw',
       'gpt4.1_TW_OVERT_nl_tw',
       'gpt4.1_TW_MONDEC_nl_tw',
       'gpt4.1_TW_ORGANE_nl_tw'] # MANUALLY change column names

# # MANUALLY select the English or Dutch function below
# #Extract numeric rating from the explanation columns into new columns
# #English
# for expl_col, num_col in zip(explanation_cols, numeric_cols):
#     df[num_col] = df[expl_col].str.extract(r'Rating:\s*([0-9]+(?:\.[0-9])?)')[0].astype(float)

# # Dutch 
for expl_col, num_col in zip(explanation_cols, numeric_cols):
    df[num_col] = df[expl_col].str.extract(r'Score:\s*([0-9]+(?:\.[0-9])?)')[0].astype(float)


# Compute the row-wise average of the four numeric columns and store in a new column
df['gpt4.1_TW_nl_tw_total'] = df[numeric_cols].mean(axis=1) # MANUALLY change column name

df.head()

,id,Beoordeling.all_nl_tw,TW_SAMEN,TW_OVERT,TW_MONDEC,TW_ORGANE,TW_total,gpt4.1_TW_SAMEN_nl_tw_explanation,gpt4.1_TW_OVERT_nl_tw_explanation,gpt4.1_TW_MONDEC_nl_tw_explanation,gpt4.1_TW_ORGANE_nl_tw_explanation,gpt4.1_TW_SAMEN_nl_tw,gpt4.1_TW_OVERT_nl_tw,gpt4.1_TW_MONDEC_nl_tw,gpt4.1_TW_ORGANE_nl_tw,gpt4.1_TW_nl_tw_total
1,1462073,"Marine, ja, ik begrijp inderdaad dat jij het d...",3.000000,3.000000,3.000000,3.000000,3.000000,Score: 4.0 - Toelichting: De kandidaat toont s...,Score: 3.2 - Toelichting: De kandidaat reageer...,Score: 2.4 - Toelichting: De kandidaat communi...,Score: 3.0 - Toelichting: De kandidaat toont b...,4.0,3.2,2.4,3.0,3.150
3,1552130,"Hoi marine, leuk om met je samen te werken. Zo...",3.500000,3.001667,2.831667,3.333333,3.166667,Score: 4.0 - Toelichting: De kandidaat toont s...,Score: 4.0 - Toelichting: De kandidaat reageer...,Score: 3.2 - Toelichting: De kandidaat communi...,Score: 2.5 - Toelichting: De kandidaat toont e...,4.0,4.0,3.2,2.5,3.425
5,1584726,"Marine, wat goed dat we elkaar inderdaad treff...",3.501667,3.666667,3.001667,3.833333,3.500833,Score: 4.0 - Toelichting: De kandidaat toont c...,Score: 4.0 - Toelichting: De kandidaat herkent...,Score: 4.0 - Toelichting: De kandidaat communi...,Score: 4.0 - Toelichting: De kandidaat toont e...,4.0,4.0,4.0,4.0,4.000
6,1584752,"Hoi marine, bedankt voor je compliment dat je ...",3.500000,3.166667,3.166667,3.500000,3.333333,Score: 3.5 - Toelichting: De kandidaat benadru...,Score: 3.2 - Toelichting: De kandidaat benoemt...,Score: 3.2 - Toelichting: De kandidaat communi...,Score: 3.0 - Toelichting: De kandidaat stelt e...,3.5,3.2,3.2,3.0,3.225
7,1586157,Ik snap dat je het heel druk hebt. Dat hebben ...,3.666667,2.998333,3.001667,3.335000,3.250417,Score: 4.3 - Toelichting: De kandidaat erkent ...,Score: 3.5 - Toelichting: De kandidaat erkent ...,Score: 4.0 - Toelichting: De kandidaat communi...,Score: 3.5 - Toelichting: De kandidaat toont b...,4.3,3.5,4.0,3.5,3.825


In [82]:
# Optional: Save results - MANUALLY change the name of the file
df.to_csv("output/LLM_predictions/TW/TW_nl_tw_gpt4.1.csv", index=False)
df.to_excel("output/LLM_predictions/TW/TW_nl_tw_gpt4.1.xlsx", index = False)

In [83]:
# Get performance evaluation metrics

# Define column pairs
original_cols = ['TW_total', 'TW_SAMEN', 'TW_OVERT', 'TW_MONDEC', 'TW_ORGANE'] # MANUALLY Define column pairs
gpt_cols = ['gpt4.1_TW_nl_tw_total', 'gpt4.1_TW_SAMEN_nl_tw',
       'gpt4.1_TW_OVERT_nl_tw',
       'gpt4.1_TW_MONDEC_nl_tw',
       'gpt4.1_TW_ORGANE_nl_tw'] # MANUALLY Define column pairs

results = []

for orig, pred in zip(original_cols, gpt_cols):

    # Drop missing values pairwise
    temp_df = df[[orig, pred]].dropna()

    y_true = temp_df[orig]
    y_pred = temp_df[pred]

    # Metrics
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    r, p = pearsonr(y_true, y_pred)

    results.append({
        'Variable': orig,
        'MAE': round(mae, 2),
        'RMSE': round(rmse, 2),
        'R2': round(r2, 2),
        'Pearson_r': round(r, 2),
        'p_value': round(p, 2)
    })

# Convert to DataFrame
performance_df = pd.DataFrame(results)

print(performance_df)

    Variable   MAE  RMSE    R2  Pearson_r  p_value
0   TW_total  0.53  0.66 -0.88       0.29     0.00
1   TW_SAMEN  0.66  0.83 -0.76       0.32     0.00
2   TW_OVERT  0.70  0.85 -0.63       0.32     0.00
3  TW_MONDEC  0.64  0.79 -0.67       0.27     0.00
4  TW_ORGANE  0.79  0.98 -1.72       0.13     0.05


In [85]:
# Save to Excel
output_path = "output/LLM_predictions/TW/TW_nl_tw_gpt4.1_performance.xlsx" # MANUALLY change directory and file name
performance_df.to_excel(output_path, index=False)